<a href="https://colab.research.google.com/github/RegmiYogesh/Object_Centric_Patch-_sampling/blob/main/Single_Class_Object_Centric.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import random
import rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, box

# ==============================
# PATHS
# ==============================
satellite_fp = r"D:\Paper\Cotton_Fnal.tif"
cotton_fp = r"D:\Paper\Whole_cotton.shp" # Renamed 'cotton' to 'cotton_fp' for clarity
base_output = r'D:\Paper\cotton'

img_dir = os.path.join(base_output, "images")
lbl_dir = os.path.join(base_output, "labels")

os.makedirs(img_dir, exist_ok=True)
os.makedirs(lbl_dir, exist_ok=True)

PATCH_SIZE = 256
HALF = PATCH_SIZE // 2
BACKGROUND_RATIO = 0.20

# ==============================
# LOAD VECTOR DATA
# ==============================
gdf = gpd.read_file(cotton_fp)

with rasterio.open(satellite_fp) as src:

    # Reproject vector to raster CRS
    gdf = gdf.to_crs(src.crs)

    # ---------- METADATA ----------
    img_meta = src.meta.copy()

    lbl_meta = src.meta.copy()
    lbl_meta.update(
        {
            "count": 1,
            "dtype": "uint8",
            # Removed "nodata": 0 to ensure 0 values are not masked out
        }
    )

    # ---------- UTILITY ----------
    def is_window_valid(w):
        return (
            w.col_off >= 0
            and w.row_off >= 0
            and w.col_off + w.width <= src.width
            and w.row_off + w.height <= src.height
        )

    saved_count = 0

    # ==============================
    # 1‴ COTTON PATCHES
    # ==============================
    for idx, row in gdf.iterrows():

        fid = row['fid'] if 'fid' in row else idx
        centroid = row.geometry.centroid

        # Map → pixel
        col, row_pix = ~src.transform * (centroid.x, centroid.y)
        col, row_pix = int(col), int(row_pix)

        window = Window(col - HALF, row_pix - HALF, PATCH_SIZE, PATCH_SIZE)

        if not is_window_valid(window):
            continue

        # ---------- IMAGE ----------
        image = src.read(window=window)
        win_transform = src.window_transform(window)

        img_meta.update(
            {"height": PATCH_SIZE, "width": PATCH_SIZE, "transform": win_transform}
        )

        img_fp = os.path.join(img_dir, f"cotton_{fid}.tif")
        with rasterio.open(img_fp, "w", **img_meta) as dst:
            dst.write(image)

        # ---------- LABEL (ZERO FIRST) ----------
        label = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)

        window_bounds = rasterio.windows.bounds(window, src.transform)
        window_geom = box(*window_bounds)

        intersecting = gdf[gdf.intersects(window_geom)]

        if not intersecting.empty:
            burned = rasterize(
                [(geom, 1) for geom in intersecting.geometry],
                out_shape=(PATCH_SIZE, PATCH_SIZE),
                transform=win_transform,
                fill=0,
                dtype="uint8",
            )
            label[burned == 1] = 1

        lbl_meta.update(
            {"height": PATCH_SIZE, "width": PATCH_SIZE, "transform": win_transform}
        )

        lbl_fp = os.path.join(lbl_dir, f"cotton_{fid}.tif")
        with rasterio.open(lbl_fp, "w", **lbl_meta) as dst:
            dst.write(label, 1)

        saved_count += 1

    print(f"Saved {saved_count} cotton image-label pairs")

    # ==============================
    # 2‴ BACKGROUND PATCHES (20%)
    # ==============================
    num_background = int(saved_count * BACKGROUND_RATIO)

    bg_count = 0
    attempts = 0

    while bg_count < num_background and attempts < num_background * 10:

        attempts += 1

        col = random.randint(HALF, src.width - HALF)
        row_pix = random.randint(HALF, src.height - HALF)

        x, y = src.transform * (col, row_pix)
        point = Point(x, y)

        # Skip if centroid is inside any cotton
        # This check is less strict than the window intersection check below
        if gdf.contains(point).any():
            continue

        window = Window(col - HALF, row_pix - HALF, PATCH_SIZE, PATCH_SIZE)

        if not is_window_valid(window):
            continue

        # Check if the candidate background window intersects ANY cotton
        window_bounds = rasterio.windows.bounds(window, src.transform)
        window_geom = box(*window_bounds)
        intersecting_cottons = gdf[gdf.intersects(window_geom)]
        if not intersecting_cottons.empty:
            continue  # This patch contains cottons, so it's not a true background patch

        # ---------- IMAGE ----------
        image = src.read(window=window)

        # NEW: Skip if the image is completely black (no data)
        if np.all(image == 0):
            continue

        win_transform = src.window_transform(window)

        img_meta.update(
            {"height": PATCH_SIZE, "width": PATCH_SIZE, "transform": win_transform}
        )

        img_fp = os.path.join(img_dir, f"background_{bg_count}.tif")
        with rasterio.open(img_fp, "w", **img_meta) as dst:
            dst.write(image)

        # ---------- LABEL (ALL ZERO) ----------
        label = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)

        lbl_meta.update(
            {"height": PATCH_SIZE, "width": PATCH_SIZE, "transform": win_transform}
        )

        lbl_fp = os.path.join(lbl_dir, f"background_{bg_count}.tif")
        with rasterio.open(lbl_fp, "w", **lbl_meta) as dst:
            dst.write(label, 1)

        bg_count += 1

    print(f"Saved {bg_count} background image-label pairs")

Saved 1418 cotton image-label pairs
Saved 283 background image-label pairs
